# 01 - Data preparation and first look

Start by checking the raw Olympic results and country mapping. The goal is to make one row per country per Summer Games, then add World Bank indicators. Do not train a model until the country matching and medal counting rules are understood.

Expected raw files in `data/raw/`: `athlete_events.csv`, `noc_regions.csv`, and a World Bank indicators CSV. The first two are from Kaggle's *120 years of Olympic history: athletes and results*.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

print(f'Project folder: {PROJECT_ROOT.resolve()}')
print(f'Raw data folder: {RAW_DIR.resolve()}')

## 1. Check which source files are present

Download the files listed in the README and put them in `data/raw/`. Keep the original filenames where possible.

In [ ]:
for filename in ['athlete_events.csv', 'noc_regions.csv']:
    path = RAW_DIR / filename
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {path}")

print('Raw folder contents:', [p.name for p in RAW_DIR.iterdir() if p.is_file()])

## 2. Load and inspect the Olympic files

Run this after adding both Olympic CSVs. Check the available columns and country codes before deciding how to aggregate team events. In the athlete-level source, medal entries for a team event may repeat for every athlete on that team.

In [ ]:
athletes_path = RAW_DIR / 'athlete_events.csv'
noc_path = RAW_DIR / 'noc_regions.csv'

if athletes_path.exists() and noc_path.exists():
    athletes = pd.read_csv(athletes_path)
    noc_regions = pd.read_csv(noc_path)
    display(athletes.head())
    print('Olympic rows/columns:', athletes.shape)
    print('Olympic columns:', athletes.columns.tolist())
    display(noc_regions.head())
    print('Country mapping rows/columns:', noc_regions.shape)
else:
    print('Add athlete_events.csv and noc_regions.csv to data/raw/, then rerun this cell.')

## 3. Limit to Summer Games and inspect coverage

The assignment focuses on Summer Olympics. Verify that each Olympic year and medal category looks sensible before building the country-year target.

In [ ]:
if 'athletes' in globals():
    summer = athletes.loc[athletes['Season'].eq('Summer')].copy()
    print('Summer Games years:', sorted(summer['Year'].dropna().unique().tolist()))
    print('Missing values in selected fields:')
    display(summer[['Year', 'NOC', 'Team', 'Sport', 'Event', 'Medal']].isna().sum())
    print('Medal labels:', summer['Medal'].value_counts(dropna=False).to_dict())
else:
    print('Load the Olympic CSVs in the previous section first.')

## 4. Next decisions

Before aggregating, record answers to these questions in your notes:

- What identifies a unique medal-winning event result in these files?
- How will team-event athlete duplicates be removed without losing legitimate medals?
- Which NOC codes or historical country names need to be mapped or excluded?
- Which Olympic years will be used for training and for the time-based holdout?
- For a retrospective 2020 forecast, which socioeconomic data year was available before the Games?

After resolving those choices, create the country-year medal table, merge World Bank features by country code and year, and save the result under `data/processed/`.